In [9]:
import os
import pandas as pd
import itertools as it
import ast
from itertools import product

# Veri seti yüklenir.
df = pd.read_csv(r"file_path")

# NaN değerleri temizlenir.
df_cleaned = df.dropna(subset=["numeric_sequence"]).copy()

# numeric_sequence kolonundaki değerleri string'e çevirir.
df_cleaned["numeric_sequence"] = df_cleaned["numeric_sequence"].astype(str)

Amino Asit Kompozisyonu (AAC)

In [ ]:
def calculate_AAC(sequence_string):
    # DUZELTME (audit fix): onceki surum stringlestirilmis liste uzerinde substring
    # sayimi yapiyordu ('1', '10'-'19' ve '21' icinde de geciyordu) ve toplam amino
    # asit sayisini karakter uzunlugundan hesapliyordu. Simdi DPC'deki gibi once
    # tam sayi listesine cevrilir (ast.literal_eval).
    sequence = [int(item) for item in ast.literal_eval(sequence_string)]
    # Her amino asit icin sayim yapilir.
    aa_counts = {aa: sequence.count(aa) for aa in range(1, 27)}
    # Toplam amino asit sayisi hesaplanir (artik gercek rezidu sayisi).
    total_aa = len(sequence)
    # AAC hesaplama
    return {aa: (count / total_aa) * 100 for aa, count in aa_counts.items()}

# AAC ozelligi Dataframe'e eklenir.
df_cleaned["AAC"] = df_cleaned["numeric_sequence"].apply(calculate_AAC)

Dipeptit Kompozisyonu (DPC)

In [11]:
def string_to_numeric_list(string_list):
    # String ifadeyi bir liste (Python ifadesi) haline getir. Yoksa hata alıyorum...
    numeric_list = ast.literal_eval(string_list)
    # Listeyi sayısal değerlere dönüştürür.
    return [int(item) for item in numeric_list]

def calculate_DPC(sequence_string):
    sequence = string_to_numeric_list(sequence_string)
    # Tüm mümkün dipeptit kombinasyonları, 1-1, 1-2 vb.
    dipeptides = list(product(range(1, 27), repeat=2))
    dipeptide_counts = {dipeptide: 0 for dipeptide in dipeptides}
    
    # Dipeptit sayımları
    for i in range(len(sequence) - 1):
        dipeptide = (sequence[i], sequence[i+1])
        dipeptide_counts[dipeptide] += 1

    # DPC hesaplama
    total_dipeptides = len(sequence) - 1
    return {dipeptide: (count / total_dipeptides) * 100 for dipeptide, count in dipeptide_counts.items()}

# DPC özelliği Dataframe'e eklenir.
df_cleaned["DPC"] = df_cleaned["numeric_sequence"].apply(calculate_DPC)

One-hot Encoding & Binary Profiling (Yalnızca, ana veri seti için uygulanır)

In [12]:
def one_hot_encoding(sequence_string, max_length=113):
    sequence = string_to_numeric_list(sequence_string)
    # 26 boyutlu one-hot vektörleri oluşturulur.
    one_hot_encoded = []
    for aa in sequence:
        vector = [0] * 26
        vector[int(aa) - 1] = 1
        one_hot_encoded.append(vector)

    # Eksik uzunlukları 'dummy' amino asitle (tüm değerler 0) tamamlanır.
    for _ in range(max_length - len(sequence)):
        one_hot_encoded.append([0] * 26)

    return one_hot_encoded

# One-hot kodlama Dataframe'e eklenir.
df_cleaned["One_Hot_Encoded"] = df_cleaned["numeric_sequence"].apply(one_hot_encoding)

In [13]:
csv_path = "file_name.csv"
df_cleaned.to_csv(csv_path, index=False)